In [39]:
%pip -q install openai-whisper jiwer pandas
# Make sure ffmpeg is installed locally (e.g. `brew install ffmpeg` on macOS)
import subprocess, shutil
if not shutil.which("ffmpeg"):
    print("WARNING: ffmpeg not found. Install it: brew install ffmpeg (macOS) or apt install ffmpeg (Linux)")
else:
    print("ffmpeg found.")
print("Dependencies ready.")

Note: you may need to restart the kernel to use updated packages.
ffmpeg found.
Dependencies ready.


In [40]:
import os
import re
import time
import whisper
import pandas as pd
from IPython.display import display

AUDIO_DIR = "./selected_audio"
os.makedirs(AUDIO_DIR, exist_ok=True)

_model_cache = {}

def get_model(name="base"):
    if name not in _model_cache:
        print(f"Loading Whisper model '{name}'...")
        _model_cache[name] = whisper.load_model(name)
    return _model_cache[name]

def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s]", "", text)
    return text

def read_csv_auto(path):
    for enc in ["utf-8", "utf-8-sig", "latin-1"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(path, encoding="latin-1")

print("Imports and helpers ready.")

Imports and helpers ready.


In [41]:
INTENT_CONFIG = {
    "decision": {
        "tower.decision.right": [["right"], ["2"], ["two"], ["second"]],
        "tower.decision.left": [["left"], ["1"], ["number", "one"], ["first"]],
        "tower.decision.same": [["both"], ["together"], ["same", "time"], ["one", "and", "two"]]
    },
    "reason": {
        "tower.reason.tall": [["tall"], ["taller"], ["big"], ["bigger"], ["high"], ["higher"]],
        "tower.reason.weight": [["weight"], ["weighs"], ["top"], ["heavy"], ["heavier"]],
        "tower.reason.thinnerbase": [["thin"], ["thinner"], ["small"], ["smaller"], ["base"], ["bottom"]],
        "tower.reason.notsymmetrical": [["symmetry"], ["symmetric"], ["symmetrical"], ["both", "sides"], ["not", "the", "same"], ["asymmetrical"], ["asymmetric"], ["not", "same"]]
    }
}

def determine_intent(transcript, state):
    """Implementation of the 3-step intent matching logic."""
    if not transcript or str(transcript).strip() == "":
        return "default.noaudio"

    # Step 1: Clean text
    clean = normalize_text(transcript)
    words = clean.split()

    # Step 2: Get intents for current state
    state_intents = INTENT_CONFIG.get(state, {})

    # Step 3: Keyword group matching
    for intent_name, groups in state_intents.items():
        for group in groups:
            # Match if EVERY keyword in group is found as a whole word
            if all(any(re.fullmatch(re.escape(k), w) for w in words) for k in group):
                return intent_name

    return "default.unknown" # Or noaudio if you prefer

In [44]:
INPUT_CSV = "./fall_prediction_audio_log.csv"
WHISPER_MODEL_BATCH = "base"  # options: "tiny", "base", "small", "medium", "large"
BATCH_STATE = "default"       # options: "decision", "reason", "default"

KEYWORDS = {
    "reason": [
        "tall", "taller", "big", "high", "higher", "weight", "weighs", "top",
        "heavy", "heavier", "thin", "thinner", "small", "smaller",
        "symmetric", "symmetrical", "both sides", "asymmetrical", "asymmetric"
    ],
    "decision": [
        "right", "two", "left", "one", "both", "together", "same time",
    ],
    "default": [
        "tall", "taller", "big", "high", "higher", "weight", "weighs", "top",
        "heavy", "heavier", "thin", "thinner", "small", "smaller",
        "symmetric", "symmetrical", "both sides", "asymmetrical", "asymmetric",
        "right", "two", "left", "one", "both", "together", "same time",
    ],
}

def run_batch_pipeline(csv_path, model_name, state):
    if not os.path.exists(csv_path):
        print(f"Error: CSV not found at {csv_path}")
        print(f"Place your CSV file at: {os.path.abspath(csv_path)}")
        return

    df = read_csv_auto(csv_path)

    transcript_col = "re-run transcription"
    intent_col = f"re-run intent [{state}]"
    timing_col = "re-run processing time [ms]"

    for col in [transcript_col, intent_col, timing_col]:
        if col not in df.columns:
            df[col] = ""

    model = get_model(model_name)
    processed_count = 0
    skipped_count = 0
    error_count = 0

    keywords = KEYWORDS.get(state, KEYWORDS["default"])
    batch_prompt = "Try to focus specifically on these words as you listen to the audio: " + ", ".join(keywords)
    print(f"INITIAL PROMPT [{state}]: {batch_prompt}")

    print(f"Starting batch process for state: {state} on {len(df)} rows...\n")

    for idx, row in df.iterrows():
        if str(row.get(intent_col, '')).strip() not in ["", "nan"]:
            continue

        file_name = str(row['fileName']).strip()
        wav_path = os.path.join(AUDIO_DIR, file_name)

        if not os.path.exists(wav_path):
            skipped_count += 1
            continue

        existing_text = str(row.get(transcript_col, '')).strip()
        if existing_text in ["", "nan"]:
            try:
                start_time = time.time()
                result = model.transcribe(wav_path, initial_prompt=batch_prompt, language="en")
                elapsed_ms = (time.time() - start_time) * 1000

                new_text = result["text"].strip()
                df.at[idx, transcript_col] = new_text
                df.at[idx, timing_col] = round(elapsed_ms, 2)
            except Exception as e:
                print(f"Row {idx} ERROR: {file_name} — {e}")
                df.at[idx, transcript_col] = "ERROR"
                error_count += 1
                continue
        else:
            new_text = existing_text

        new_intent = determine_intent(new_text, state)

        df.at[idx, intent_col] = new_intent
        processed_count += 1
        print(f"Row {idx} processed: {file_name}")

    df.to_csv(csv_path, index=False)
    print(f"\nBatch Finished!")
    print(f"Processed: {processed_count} | Skipped: {skipped_count} | Errors: {error_count}")
    display(df.head(10))

run_batch_pipeline(INPUT_CSV, WHISPER_MODEL_BATCH, BATCH_STATE)

INITIAL PROMPT [default]: Try to focus specifically on these words as you listen to the audio: tall, taller, big, high, higher, weight, weighs, top, heavy, heavier, thin, thinner, small, smaller, symmetric, symmetrical, both sides, asymmetrical, asymmetric, right, two, left, one, both, together, same time
Starting batch process for state: default on 197 rows...



/usr/local/anaconda3/lib/python3.8/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Row 41 processed: 2026-6-14--11-38--16.wav


/usr/local/anaconda3/lib/python3.8/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Row 93 processed: 2026-6-14--13-36--3.wav


/usr/local/anaconda3/lib/python3.8/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Row 95 processed: 2026-6-14--13-40--5.wav

Batch Finished!
Processed: 3 | Skipped: 194 | Errors: 0


,date,time,duration_ms,filePath,fileName,fileSize_bytes,format,sampleRate,transcription,confidence,filePath.1,Process Time,Intent,Manual Transcription,Speaker,re-run transcription,re-run intent [default],re-run processing time [ms]
0,6/14/2026,10:13:37,2373,C:\Users\noril\Downloads\processing-3.5.4\data...,2026-6-14--10-13--1.wav,41006,WAV,16000,OVER 5 SECONDS,0.0000,C:\Users\noril\Downloads\processing-3.5.4\data...,0.000000,default.noaudio,NO SPOKEN WORDS,Norilla,,,
1,6/14/2026,10:13:53,6036,C:\Users\noril\Downloads\processing-3.5.4\data...,2026-6-14--10-13--2.wav,153646,WAV,16000,I think number two first.,0.5069,C:\Users\noril\Downloads\processing-3.5.4\data...,1.278979,tower.decision.right,I think number two will fall first,Norilla,,,
2,6/14/2026,10:17:51,6094,C:\Users\noril\Downloads\processing-3.5.4\data...,2026-6-14--10-17--1.wav,163886,WAV,16000,"19, double, one, all, first, power on the left",0.4500,C:\Users\noril\Downloads\processing-3.5.4\data...,1.742168,tower.decision.left,I think number one will fall first tower on t...,Norilla,,,
3,6/14/2026,10:18:28,6037,C:\Users\noril\Downloads\processing-3.5.4\data...,2026-6-14--10-18--3.wav,163886,WAV,16000,I think the tower on the right side is for focus,0.5904,C:\Users\noril\Downloads\processing-3.5.4\data...,1.345382,tower.decision.right,I think the tower on the right side will fall ...,Norilla,,,
4,6/14/2026,10:22:28,6026,C:\Users\noril\Downloads\processing-3.5.4\data...,2026-6-14--10-22--5.wav,163886,WAV,16000,"I think number one, both first, the middle top",0.4423,C:\Users\noril\Downloads\processing-3.5.4\data...,1.483369,tower.decision.left,I think number one will fall first,Norilla,,,
5,6/14/2026,10:23:41,6014,C:\Users\noril\Downloads\processing-3.5.4\data...,2026-6-14--10-23--7.wav,153646,WAV,16000,and I think that's how we're on the left foot ...,0.3876,C:\Users\noril\Downloads\processing-3.5.4\data...,2.133826,tower.decision.left,I think the tower on the left will fall first,Norilla,,,
6,6/14/2026,10:24:19,6074,C:\Users\noril\Downloads\processing-3.5.4\data...,2026-6-14--10-24--9.wav,163886,WAV,16000,"number two, tower number two, cold wrist",0.5248,C:\Users\noril\Downloads\processing-3.5.4\data...,1.426940,tower.decision.right,The number two tower number two will fall first,Norilla,,,
7,6/14/2026,10:24:56,6021,C:\Users\noril\Downloads\processing-3.5.4\data...,2026-6-14--10-24--11.wav,153646,WAV,16000,I think the tower on the right is tall and taller,0.5028,C:\Users\noril\Downloads\processing-3.5.4\data...,1.447434,tower.decision.right,I think the tower on the right side will fall ...,Norilla,,,
8,6/14/2026,10:26:08,6050,C:\Users\noril\Downloads\processing-3.5.4\data...,2026-6-14--10-26--13.wav,163886,WAV,16000,OVER 5 SECONDS,0.0000,C:\Users\noril\Downloads\processing-3.5.4\data...,0.000000,NaN,what does it one one?,User,,,
9,6/14/2026,10:26:23,6112,C:\Users\noril\Downloads\processing-3.5.4\data...,2026-6-14--10-26--14.wav,153646,WAV,16000,number one.,0.4945,C:\Users\noril\Downloads\processing-3.5.4\data...,3.411211,NaN,number one,User,,,


In [ ]:
def run_intent_classification(csv_path):
    if not os.path.exists(csv_path):
        print(f"Error: CSV not found at {csv_path}")
        return

    df = read_csv_auto(csv_path)

    transcript_col = "re-run transcription"
    decision_col = "re-run intent [decision]"
    reason_col = "re-run intent [reason]"

    if transcript_col not in df.columns:
        print(f"No '{transcript_col}' column found.")
        return

    for col in [decision_col, reason_col]:
        if col not in df.columns:
            df[col] = ""

    classified = 0
    for idx, row in df.iterrows():
        transcript = str(row.get(transcript_col, '')).strip()
        if transcript in ["", "nan", "ERROR"]:
            continue

        df.at[idx, decision_col] = determine_intent(transcript, "decision")
        df.at[idx, reason_col] = determine_intent(transcript, "reason")
        classified += 1

    df.to_csv(csv_path, index=False)
    print(f"Classified {classified} rows for both decision and reason intents.")
    display(df[[transcript_col, decision_col, reason_col]].head(20))

run_intent_classification(INPUT_CSV)